---
title: "5.3 Getting public data by code"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/05-workflow/03-public-data.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/05-workflow/03-public-data.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If the GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. Each notebook starts afresh.

**Time:** about 35–40 minutes, including practice. **Before you begin:** Complete [5.2 Folders, paths, notebooks and scripts](02-folders-paths-scripts.ipynb).

## You will learn

- Read a documented API response and check its pagination.
- Separate an observation year from a retrieval date and a data revision.
- Verify a frozen input, preserve missingness and document reuse rights.

## Why it matters

An **API** is a documented way for a program to request information from a service. It is not a guarantee of open reuse rights, a complete response or a permanently unchanged dataset. **JSON** is a text format containing named fields and lists. We first parse made-up numbers, then inspect the actual World Bank API snapshots used in the guided projects. The country-question workflow in lesson 5.2 uses the same principle: obtain an exact public input, check it, and keep provenance beside the analysis.

## Worked example

### Inspect a tiny response before making a table

This text represents a made-up API response for fictional economies. JSON uses null for an absent value; Python converts it to None. Parse the text before selecting its records.

In [ ]:
import json
fictional_json = '{"year": 2019, "records": [{"economy": "Northland", "internet_pct": 60.0}, {"economy": "Eastria", "internet_pct": null}]}'
fictional = json.loads(fictional_json)
print(fictional["year"])
print(fictional["records"])

One value is missing, not zero. A key such as records is defined by this fictional service; real APIs can have different structures. Read the provider’s documentation rather than guessing a table layout.

### Prepare the pinned public-data snapshot

Run this prepared setup. It uses the course files locally or recreates the exact frozen public-data snapshot bundled in this notebook. It verifies each checksum. The source observations are from 2019; the snapshot was retrieved on 2026-11-11. No account, key or private records are needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared project-data setup"
from pathlib import Path
import hashlib
import base64
import zlib

# Frozen public-data snapshots, generated from the rights-documented course inputs.
expected_files = {'raw/wb-internet-2019.json': '1c576afd87af68e35ba7142745ce821879337aa7812d860ae87b324f983f3a87', 'raw/wb-economies.json': 'c9acf3932785191a649cbc04c3056af4cf64a236e5c3b49d49be25ddb0440bce', 'internet_urban_2019.csv': '95a88e1697502cff165ac757b29e8bac5c361c97615e4701b1dbfdd4d6a839e9'}
saved_snapshots = {'raw/wb-internet-2019.json': 'eNrVXftv20a2/leIAHc3BVxihvPub3pZUiTKikjJj8ViQUuszEYiXYpK6iz2f78zctsMnZxxcxe4GAVFm6T2WN88zvnO+x//fvOYbfM3P+GL028Oz7/L63/9/tcIoYs3TdVkuzc/RZxfvDlUx3qdF5s3P72J3ly82WWH5vi4yZr89Dco4j9irP9585+Lf/z7TVFuinXWVPWbn/QfzFeM03A2SMNlMliEd4le4GO2O+bmf+gv/VhsjtnuEBwPRbkNmoc8GJdNXpd5E7z9n6D6OXisHo+7rCmq8gf9A96sq2PZ1E9/Ln43shbs/Fzrnx0M9AfUSwRZuQmS6qgXrUvre4tDRdbV5vQNlwP97QbKCQlWXxaLcKihH8ui0f9L/3V1f/jXocma4+H5z5t8XezNFqH/XPz/ox5/jfo6/4K6l+sv1x8OAH0NgpYh8xY07tig6+w+uK7q3QYAuehCIMvjbuctyIRYC/ayuri/z7MyOOgPtAvMp9Tv9ZuAe0lyloC70gb8fG+DwbGuHvPTVTYrdrNdU6wh4IPzPOlVZC04yOrd04+bfF9t6+zxoVgHpx+Ql8D1HnQWZwn6jrZAH5qgcyiy4G/BPFsXPxdrCC14t7kIsbdo6cCNNnib/7beHTdm9Ydi+xAU5bra5z9AuzAHd4F6uwcpfW0Pxv2O/uO4u+gHz99c5AdgC9JBB96CkHi7CTc9exO0cAuyOs+AY46X5/m0xQuMWoD/7Q8mcjp0AG8PfNwSeXymYvAa3u983j3wbguf9yEVr+/Ddz3xHrgNLIy83YbB8qtt0LxtWeqvB857eZ7v/NLmLZd1ti12RfN0EfSq8uddsW4uTrRtVVS7vFznwdvL3uqH4FA0x9OSAInTX3SWm3FjC4FRnn0sdk/6mW/ye20d62+u6i8XPng7Gs97wLUfzXvnuQF9ewO+CLlvowQFvfDY7ry5tBc0Mqwqd0/fBjju9iGMjHssxO9Se8Ev0vrZDQRATSGoVITS3+McvYB6vwPNrHEftC0jHip/MY5fYHTc2P4NCDHy+RiHLyC6rmofpFYR9ZhTrGxn0FR//O/wEEzTwXmaEe/amIsy6Ozzk4dTs8o/HGIA5t7M4Sfw+C7/JczfZ05MOz3HXvgrutLIsRdmOWs/vsesmHYc2yH8vRpTezty4z/Z5B/znTYvNl9A/xQsZ8F6lx0OxqVyWg24Ff2eI+DhryS8ubK3ofqkj31fbDa73Ek3p/HteZLq+AXa/wunxjz01yt4M2sjzOu/cp4QVEI8hnr33lowfgZp/KAXwayqm4fgOXx3of+7fcjKQn/u8uQd/XD67bc3Iob9oNTfqOXNf7UT36f+4hm8Q9hjYzMl/90efVGJkCKMZ46N8diFPv96Y1yyIh6fqUPF9iP+fubP/OfbOGcwz5PKZ2Jz28LZ/Elf8s33CX+vT/PK9g9eDXr9YJ/v7/MacINeDfrnmbVgB/muTMLNX8hYuDrTjIXEtlD+iGIWh51xe7+Oen6mqFf2Ec+rQ/Mdvoh5kp4nZmxjrr/H/TJfnKf7JbEhJ6/e5gQ+WR55nJkh7QVPOYKOCHUCp59EPqvYlH0T5F8hiUnnLBHf2U7h5Hj/Y5I9ZLXmxc+0GbrDlw7vsMeG5aUT7PfZS4njxJnP2Qf8lU34Hh8hrJq9vgg3dvBu+fj43zlTuM+xWHxtLehI/72ewnFY4rH3u3PZSuz+08aHMrmHoPdPhByZX8TfsEfH9m13dvcaKyCiO1MwDstlyBgiCuMIS4T89X7171pot9qmB9D270BZzFgoFOJKIX+5Vcde8HffRRkk2b6CTjeJz5Ird+zcl065qeoaQjgDxZFCocCSK848lkpXLaDbagfhHF6BPnptDkSKKEn81S6dYQtnU2yP2SmbrZvV98cNBDoFpbDAIVIRU0L5K5dOlQRfymq2ucZdQlAXMFQVKsoVixChgjGPxVPcwrvPYa2zAOUS5yGjRFHFJefY56d73UJ7vIewdq/PUwbb7vLO8WCSjsHzXIL8XhFNl4xWpf6aOp30JVQYaOrItBQsihAVUmi25PHFbdGlz3l9nxW/gET4DvS2SSOCzS/EGWbUX99F116wq63YfXa4CNIHwHzrjhIHpeCKRFrV+muzdkdttHVWlBBQsNRNqVAgzBAlEgkh/LV0uv0W2nK7yzb54QEAPATJIkEhkQQj5LF+7XZbWPW73VSA97gLF+makhckkL7D/oqorh3C7Oa7rD5CQKfgHZZRKKTCmjcpjHw267qDNtptcdwDaAdTh2iKBKORRNpcpx6DvWuDLT5DYnh6B9rqMhRcnytmHiuddy2cJSiEB2DWSIRChoyKpf4adt24BbPeg8Zc90yrL7s2Oew+HEGfYTcFT1LQEAttogvmsSa1XRHdaqcXhE7yauqoT6GCSnxyGnLlb9prt9NCe9Cm6skfMcrrz/m2+gga6t3xCDRdT4a6NtI5weLU38VX8Nct8M3hUwbCvYaz/WQoo0goban7i9R2wXTr7HOxgwgTXCNMQoUjSpTChDDpL2taDVtgi6Y4PASrot4WZTA+ZdEAJGo1PM9GH91ZC/CxzIugb7jiIdtlEIVagFJaeaxmW0d73G0z0EXRHS5c5SmUK4kUxYIjfx0y3csW3PqDFsfBZXaoAMiX4ONVmhVjibjmUYxg6bFMHrchH/WaANr+GDzgEIkIKyUjfw2AUz36lz5M91WwyusNYAT05mD1Osf6MhPNqQRTBEl/ddBk1AK8v6820NudjEC/OIu02c6k0CqXUuUv2l7cRpvXFVQy1YtBSUV4yIgyN9lfIdXrtJCWGWT59DqwysGGWygUeYxzctvC+bTPXqETvdvzjDr3Lr/RMO0526kMFvnj8X4H9dHqdcBMN6GNeBUhrYD8VT6p7UXtPWRA7k/a68MoOZYnzeMviXo3bKMsy3z3yk0ejc/zJk9bSIsdpF5HoCUvWYgMScQeO057szZMyIjtjWaO5mZIavaghbBg2N+727O9M71qV+3vIRLRg90zXB8q4gohgZmiymOd02IR1b6qoUhH7yp21OkSSaTXMZ1eS/JW5ba6CPr5PjT6JoQQg1IYkxDpi4w0n/AX8fBrxC6wQ8fxYsyJ0haAv2AXLbD6cwULMF2+txg7Yu0RipjilBIWeZxc0Bu3ADd5sPn7+GNV1JAGGq8cNkAkpWKUccYi5K9sHrUOua70VwInPFqsHFleSCghCRdMMOZvfkHPzgvqgWlPvWXX5YYSRutyZjJH/A0S9K5bUOtsnVUQ2vNM8uq1bLunRzDO3rsFW+RKHiJOeKSUNgGUx0moPTv03Pucrx9ACuXIekKh5FxRSjFRPPL38vYn1oL9vNxn9QcgK342Af0TMkSUU5N4GmFG/dU7fTvc3v+luK+ODeA/7b8bOxILqODaWPeXMPZtZtyv9kUJ8ol+7IppEalxEo9Tu/pX30D6miOmD5sDWsMKc4eVx77iQaud9fqYbfRPBpoag+kUTPnczdem/4Pt02NzEZxGTsA2wGAINptiQnPESAtkYlKMPQ7gJXbUY7ALkmz3ET7cZAryRIZCqjBWFHFBpMdXeWj3JBr8ejQ/uch2wfBYlFCH9uHsPZhtoEJMiECYeHy3bUNgUBcN3IoeNvWwNvUiIrRZ668SGrQ6tB+aCqz0GMBtDEzGYiRVZNzGQmF/uUVy10L7ybROBLhFcg0mLVISKkKkkh673QZ2rtugeSiqR/BgUzAPCrNQSWqKZP11zFza5OIyq6vcHQm4XFydZ095mxdfFr8A9/YS5sSchBEWUimPBe/luIWyNAcJAB3PHE29ONI6lWETDBD+yqPLRXtOQLnOoUsLGgBSGwBECUqRJk/S4+jz/LIFNi/XD8G82j2VOdhVZn57eZZvdWgnFAyzeyhvYtjpOqg/8z1qN4xbME0gy1GNNYxBsASHGMtIKeoxC7aZ0jCv6i10a4eDK0cnBq1PBSaMY658NtnbaOt9VgLN8fsD0HyVUjN9yjRSpFkh9jdrYmgndg0fwBzq4ajjGASgVY6/inVoK9ZhcV9nuyYDjNbh+DyziIe2Oh3WeQ6p0+HCNUBJv88Ica4IFh6XJQ2nL8A6uNJwMT3PA+23McKZecNF3zX1DkdCCeavU2loh+KGRyjlfbg8z3y8YdqG1+T6o0AnmYIYKQ2pJkRUSc3tkceydtaC6/CSwTZMREKpNFSFPeZ/11/h/LFbHA7ZEXIKdh29uFSENVyPsw6Hty24TzBNWN46mlNpYcT1DfaXDY3s1zrKCij8NkpBVwNhoeAU6dvr7ysd2a90VJWbY50BPqMR3JuKqJBoq5tFWIsk7PFjHU3acLfBxPwr6SwuAkfG5WgydKS8C0qo6Z+nMJP+UqWRrV1Hx3Kb1YAxM1rOXINWhTZlCEcMUenvvR63FlznMC0cJ1NH2xCNUlFsEAuPz3Y8e7EgcI/H8BOONNaIMISov7la4357wcrhNRv3Z46qbm2McyqV0EJL+Ktrx7YpN66z8uLk1d8Xa0d4eQxXhIoopAxpQiW40Madx8Dft4H/CkF972DJJOKEcsYwZh57v8eDFlSXoILtVylC08KVsYhGDHs82WdsO0v1Xc7N98RQC4oxPILGa0tvPG2hrLMcmjeZLBxpiFrjSCE4loh5HNEYtybCNhk4PzR1DdbTvFFgFElBfTYK3tnX9122z8BktXed2NGxSlKMjOubEsI8RjtvoX2Enum7OVwQGoWCIKIVLaZaOnlcTGcH0N9V9QYEe7VwHCyS1DSa85dCTezkj0n2OfvwALcNn3TuHBUcRjaZ/pCUR9JjwLZ+neTlE1Sr7mhcFYWca7SEmKJQj/OYJnZ4Y1LUxX0GuS0m44VjuptQkVIq8tfgmdiiaVLVefZ7ndk8rx53+d8PDoI8X0zOklVMFl9jhlFOYDmleKgpsSbFWNLI47aJN5MW4EP1EShXuZncnOeJ2q7jyfFTpj/ltw/zOnX5KGjEBadMH6fH7Zwmdqb05KnePn1+Jfl9MrxzhLP8FcPTTmsydRXM+wto9vaVw0mBSKTFMPJXt05X7RHcYIu96arjsHMwMUWRlBKPe2pPu63x2vdZCU7O7s4cpIlKrP8O+6tZp0kL6KFqHgCpO03gy2u6+2BhrBvNIjw+VVulTot7eNTKtAuqU6xCqZhpHOjxS71tA30CYd46RlRQRJGSHrcfmI5bMPP1Q5OXhyaHutVOxwNHo39MOUaR8hhu2oLbPBzhyUjTdOkQS0xiySUx4UqPvYdTO4Q1Pf6W7++rY70FAC9BUqhEaJKbjHQikfA3iBXbTonYlGu/GqmMHVOeeUi5tuyYQDxiHpdaxcMW7E22zQ5rKE0v7sOh2ZAIRRGKmL8aKL5uYd1lnwCDPb52lVghJjlCHvOn+PYFzicwchffggMrpKb9UmDKEBIykv5K5njVhqtXhCbjxn24iYbQL1YJRLi/fDGetpFC13c6dozMVBwRfX39dQ/HaRtlA93daeroQCaEZIproSQR8fjujlpY68ODGe7sLCeL4cZrVIWCCarZsb+uiXjRQnysiwYmUvEidQzARabUyuOSyHj5FdQCapISw2OwOA5FpBRjWhYT4jFljG9svPlvxRqwZuPBjaMRPicRRtqURWb8ir+VdHZ4Mi7W9XOCzEVwmW/CIGkOgK/4Ep6rSUxzI1On7nFRUmynBcXVblN9hN5u39lCJNL2HvG413TcawEtM/Ay967gFjj+wpu14W0rcEJfPANZP8Om85jSxF9GPjeFjwdttE1e5tsaOtDZwHFxqeCcayUrJPfYWxF3WoDrag1e387CZQFEmhATpG1Z7nH5Q3zXQvvZFIb+egRKseIrMNKBI3286qRr/cXa0jtPmWnOBQB190inSFt2HrtRZ/YVnmX7AuxaO4PzfwwjFkwh7HFW4mzRAlrlkGCaLZaOKVGaOUjlc4XvzM4jmOWPGZCoN5uDBg4xrQc4RzjylwXPpi2YeoHaYc3Npn2HVzyS2ohFHFHkc+PWWa+F+FPQy3b5Bm7sM+udZ3Ho7O4Fzrs8g1OIZ3dTR7sQyTXRRx6XOczs8M5M/+Q62x6hAx33HNVKCBuO6PFcydmgBXWbAwp1Nli4Om9J0yDQ4/rt2fAlTPCBDjuO0hVBqRl267Gb1M5lmlV18xDE2dolkeJJ35VHgLUcjkxyOPGYL8Xzl6j1N2vg+pTL7BWX4mx+ngL5qg35Uwbkw88cCXryVM5hxlUzgqm/Rs6VTfyv9lAe8RVcx2G65jFFvJbFc/vtzrMPBZwxPe9MHMXdZk61pv1Y+dwGZn7dQruD6tjn0/PsET7vtPCV2R7quwXPueIkNEnvmAqh36fHCTHzYQvs4zELDEl0tWOYw641bGBTMxbJX0U7v20hPjHEJygR/NbRoso0HtNnHOlfHmvYuc0T53kNPVaYJjIVKoYYItqkUwh5LJjsuOT8odgVj4/6GgP8Ye4ISZqxMpxjk/KEIo+9MHPbbJ9XsF03h2ckSRRSYgx2KbjkHudiztMW2Lo5biFvzBwOwQoWEqrtWJP4I4XHjTfmtodtfszrpjKzdKrg7TL5AYI9dmSKCG0IEKUteNOHw1vY723t+z4Du6+977gqHjiLJKIRkpx4XPGwsE2BRbWHcwoWV0tHeOcU2mGCUUY9rtlf2JkFi+PhoG08E3jO69P3AMDhFAMZhZxGWFu4XCDu8VD2hU2YF5+0kIZOGR4+H+GQmBRyj6Oz1/aCSbavAJTXyXm2Z0viFr5nN0UJhD2SePFKMbtpCOQv1rSFtQrSap8H+uYGc415XTwCccoknTsabkT6BiMU+UuWk04L9nFTnGZ1QFI56YBSWbFQRFQQxhQj0mOakdg5FYlJMYBIVQIXQGvCTITiXs9wWLTEU17Dp7rouvJtMdc2PY+wkh4nFyS9Ftin9UO+20GWUHIL93nlISeaR2GPxxEmthGUFHldZ8E0r0pIRE0HjkEOApsQV+TxcI5hC225zR4raNhkMpw7mk4rqiIWcaOLuMc3+aaNt9FqN6ubvAze9o/N+iF41H8C7KHk5kyZhu1PTnbVx+zDK5XQyWrioMmSmfwDhsxcC4+V7/gF6hw0h5LVzDHoARsLV2jxzIXHdbSJXTCcmAnXVemOeCXTrqMDCZVSmhZf/gK+agHWn+PPzGO4S0UCz7cznYMw1ZCxvz6ruxahrI7NQ9D5uQabQt11Ll055opLr+NgSfIV2uQINkpKErgZueku/xzh9Hd2VAvsYwaVEA+SuSO2KbDknBGjiD0u+W/po7oIpln5ASojnoA+DBKFKELaAvTYyTxpmUNNGEyKpjmcLN9Z/rEAJPNk1nG8W0qotow87uM27b0APT2uwTLxXsdR84OkphnYY6jx5QuosaaPhSaTv49acrDJuHOeY5ZWL093VZTrXBNpc6nNcs8jI+BI2aqXuuYvCU70kXtMtOzqH5c+6p9nM9Fk0cJXF2W2hwzCJeyV1MKZmPbOHuuhxI5lJ5/yTQ4d5TXctoOGVBHTSwiJKPLYhdMbtcAWzefnnGxggvvolUYlSHLFCMPC4+O1MzOSJ5MD+OeoZJfte7s4y3eb2qMq0+yXwpU3lr6buLpPSi2DUeRvPCG9a2EtP8OhzvQOJBiYh0wZAulxxmNqP9v0ISvgN5vCc9I4NyFsihVlUjCfD9Z2vqbFvqp/nOaHBtA/6RSM5kYmpdUM0fU4NyG1fa9ptQVif+kQLK6NtMUXCeJ1DVt61UJZbqGHejVzlNgybjrEemzppXaIM9WsqdhkmxMnTqv7DDzb9MpRzqZMKgJW/qrY1DZvU/0hwSY7KTzjxQgnZVIRCNIiyuNxIOmihbb+UDxBggkmxoKESuhjjSIpMPN4dFEav0C7z0sXo5icZ3Ak7b2A+eyd6WXFujq4Hegp7LvwG/KqBdn8DrrGq7MEuLQV63ILZ0gt4SIvbDrOmBQpj71PSzscsPxQZwUUpV5OXFMEcEQlpabmP+L+atiOba4v9WfMN8/23GBf1BoZ8Eo7i4Ejs9NMP1RCH7LeA3+hD7tfQ5/opTYVNLMUbh+ropBhzk+lmgJ7nDS2TL4GnTTwQS+TjqPYmGp2Iak0feg9Tp9a2j6LZX2Ei0mWcDGJJCFhmBEiOBfM40ZKS9uMX36+z10ui+Vd1yHCyIlBcmrmvvgLeGXnLK+y8qg/F+AkX6bnGR2whfQqL/PPx3xnBkd0AZyOLEAVmvYsSnrsnlnZBtCq0N82g8ZIr2axo/qLc8wkihATxGORvBq34Nbb4s8ck+DtMkxCINa1Gp+nT3VuL3idH5qgm5UfTgbCMPsMFTQmA4eg4lHEpEkKxMJjQXVrP+PbXJt+rtkvt4PYUYVMFTLDffx9xHe2pXtnGmdB+TQxPEqbhlRgabSQvzjtwpG7Yn+f3X8C7IU7OOIV8ZBJSRD6q26pf/7zfwHOcwOJ', 'raw/wb-economies.json': 'eNrtfVlTG8uy9V/p8MPZ+0SYip6H86YJMUhCWwNgbtz4oiSVpTKtLm4P+OIb579/1cI22VmNjIS9OZh6cRhsNU1mVY4rV/7X/727oUv27l/W+81fsvu/sfT/3X/7nWua796/y0VO43f/siP/3+//6//e8YX8l0bzQv4Lz4TdEovyvzbKrxO63vw9LWZUfpmyJRfJu399/UyvNfj6mfn9Z65O5Ne3NC7KL3o054nRWLOUz6nxD6NFUz6bMZoY7/79/h1drHmCnld9GHhU+QGezMWa9dgti79/4Oi4Vf3MZRt86ogvV8b9x8oHxCxZ8GQ5ubthD7/B4BI94BI8YCByYx7TLOMfOVuUz5jTGy5l1+L5nfz3s5Qmn1iW04X8UCySJc+LzVMOApOYlh+U35ZS+PpdyyZe+c1/v/8m88NOVeZXR0DmHzdy69AsZ2li0GRhjEWRr+QXqiYGjeqvsfn626/RWC7lf6e5PA4/S/A7/Lw6sT/686ofrUobybgqWyjULjrIh1CoyxVNuFRZjRD7HfRbXf0FXq3PF4uYbfTx3hiINF8Z9yp6b4CnynM+pNf3P+ARYfex8C6f9WOMP9n/zuOilLGxejjx/3xMdT3lzvThpRWff3Bljtv4yhyDzx+3G6rmTumsiJH6/IhYgW9Vtei4xLPtECpzhJQZ4RuiL8PWy4Cs+tWxYmEu2IOFabEkT2msZbpNpt0zdCbPgEyTpYhrzuR4fIhsSxe81biYHYzpikp/8vW2P+ojx2NspA63P2hX89DH5mFQNQ8sNdb3NuoHdqLZRg+CL3rcHLVVDfQKeQgp0oPlENu1q9o4CEloWZ4HtNJrIq30gFbimTScNWrptMZImgF4yU6RihtWhi7318JoZPIhjyim00KKCTo/etSuupkqupmAHzG9ufmVuplweagY1k0k9RAg5bgWcRwLxjmDNtJNG96YhUjTX6ub1xZYfpWJEVPjnMWxciOkl7RCJHQZXJoBvBAjdCGsBgzo6cy4EGm80KZ+m6kfoQC90XmQ4TThOVsYG1F21vJ2lD/ppYLKV3fCZ4XRXtEZR5rwXOIEpldViO0SN/BdqBcc44/g2V5Kk8CTvyFh7TWwEJ/2vNdk95sFS0RmNHi6Od6VNNcLiWsFDvLNMor3LQtG8aM+UlcfqmvNtG9+no4+sJTdblJaqB5XZlNmhNyESSwv8oFuxlg3Y6Cb+7ObGGO6FnUqamAVuVCu0qrdy7K0X3NpCea/i/ka0qUwyj/wjbACk/gRSmsPLJfYoQulPsEGrAsjopwvC7pJyJo0nRWL37L2ts9BH1Oe5MaJWCV/KMbIt0joej6quQXEsiph0XSMJD8Fki+y0gBwfdQfRN6iyYzdB+iVGNSNiGVH2PR78qDbUNoTJO0JkjbXkT+U9jlnSaLI2pdBUYQD/pDYtmkBWV/hYPUKyPoLS2eUf6qre2pH+/RgiF4X2MvKFDhybMXNOqEDItZm+7iqnCYoxDWLtJDv95qrRn9/UblZfCrWs0IxTHZEHN9RDJO8QSHUx/e07Js+3Ad91Fhz48/yQByIJL4z5qKQx5ez7J86ed6SPDc7LSTiAIi43hRoIe8s5B4SMqhQNFm85MVaO1hoNtIiy1iMo0dXGg2lHeWZJHQCC0p7gKR9AqWd8ERX/vcy5kOR5uJgIG5xQmUT37FRgcEnbhBEQCuH2JYfVnzrNU+ocUgzoR3sLjo5K+iSLkSxFIWS5RLPcSJPgTc4vglSrWYXlf2boOzfpMkypguWrWq0gnOtcAyFWaIgttqkXT7/n3YT2it6jUOayCSuZaFbYDskML2KuFGvvNmF1yBe0t8v2dqrhiA+ciVolPGh7agtlcCqHOgjHMRbDxIuf9hD1AJ8kEEzeMV1OLMtnDnCZ/gImoxVSnmieyxPDHX6VAoOn3QZ03iO5yJb4hMrsmGgc4SqZM1xRQ/yudl7Y7JiGhSIpT6Q/0QVjxkExMFJqe0R0/eh1I+PkNRBy7YpsoTfF4WPWPqFLcVtbY9LV3N2qCan9BNTYk4rJK5toSviOiT0TQBqaPZw1Gk/KAsf83zFYPtPp7m7+oUe9gsfKmkulVmdvgrPuAp9nmTXOCgKiBf4qKHiOSTygxCq5gqp5qqiGv6F6Ua8mhg00YOOwIOa5UdqShcsXosbpc17EIYkCHB6YAXE9ioV6D6uXTgPetoDdbyJd0sbpi3XVsvVnyKx9+H1SNe13d2Bcp6nlRBko6D78/y7xE1HdM3jXCin23dlnouMkGPLdC1ygJTPcBn0DEZOMb/lvw8c6GUhulTagC+KksoEwvEx7EH6isiEvmKEYyaA22qm9AuPta/4KX3KlGY8VsocB25AIjvCYGrLI0EIW8lNjB1tNmH6l87oQmQak/JN1nyxZLn4rJguLyK+hcGM8lKYkQOrSiPsmAfwUhQJ40a7DHAzGtO1RqaAZnyyoKkxlgfKaEo3/VkJjSzLJZGLzFL5LSuC/YMxLnb4DwqoK93r9G3XIGiCjziAATVXRe043lvuAkxWfH2zwlWksLQnAZoqvc/SoJO9wE72AsZCeSbvyWueUHpRx9qlM5EKZQ7G9qRfNTHs03aJ77kgB2s1kGpa0Q8szWHruwmcyyeLRBuabYam1ThEAgat4O/FmY/3oOYRuylmsfSHui28C+6KJssCj23YFvFdE1kmj/iO6flQO8gNtEB9uyWtkk6F7zvveS6DGaWF4JXIcozn98paNYgmWxjY1qoFtpVthLIo/QBv05ZmN0vTwSlSqFqar3Xhb8Ju0jjn80zLdatcUb2s5SmoQTCp/g03qE/vblI+QljxFuizjz/z/AtLYylmDR2s1N9TnGAGxHVDG8V9ri8zzBAKG8FGTgAwp7WiScJi4zgr5Z1pge9zmFHxt9WD8uUx06Wqbw3vJOc1U2vl0JobeBgz7xAXjk61jnD4NqjIOfmVY1OdxrD6ILez/UGvq17L+Cf5HKV85RM79JXREtONoFqOz5FawGhJS+TMWPxxfCt4yjQ4ec/O6we6FkWWieI6Ve6OS0wH13g94jhgCq6FoSIt/zGoyLdQEYJFdGyzmzvo49gmqG10b2QNe91a1LuLGsFyWn2Y0MszLYQeidjf6ogiWSiUOxbxrACh0xwSBhC/2TpD0PtWG/qEZCneG222JmUNjOj6105kdjzJVjRTZnSlzbdt3Fl15Tehqz5DY/+trqKW166RF70yJYbgC73lccxU/di+r+rHDiAjZwvjSFpnUEGxWM+45pX5OSGvWIpcLXLKcCrEw9TEN6OwoiXEWXIK3Y5Yi7QOnaAH8Z5g3fpldwtX992SgQ8Tjh1IR+RHISzvD3Eicg6jgZkwzlm6YLr1uKdxG6ZURfTYDvHMEDMNuySyIQNTa4RKUK0RvDFZTo1RLZfqm62TGCciYwrSVlonM0QtmIj4TgDRU60xTvdAa+ABd6DTjZ3TDQzUGTvQwHw7e9maxrEhz3QtIZ+W54M8p6iH1QJcSK2/hfL9tUU6z8U431KF3ecgLAenMVm8tOuW4zhQWYjKuXUBlZXSORV6Gg9L/KJMBdY1FP2P3okPOLb8AG3M3VoK59FezVuXdpeJdMmMSQ0WNrSIE3o4TomIY9qwUvphiA45lP7dza8e83p1w6Z8LjIlKHRKtiOMxnc8YvkBDNcxX1gLDG61vrD5SnOz4fh7WSh1BZe4tlL/N2UWC4fk2hiV0/4h3VS7ccBivuQz6ZM0zPhJtruNGafaT2Cc0nLeQ84Io9MGpFNdlkofeactRxU5EvNE3WvgWha2HDbxbDgd0j5BmXsbUE61P/GZzCf5m91l85JEPQ+yR7Uyy7XxzI9FvNAEvreNO8LtHcgDtMXa3WL1sbxBxbgt5C//t1S/3sSY+khkTCVA8WX8jwpnZUsmghFpG4+ot585ot5uGJtdd/pybL0cg1Mk9lNwOViypum1dueQElv+asmKLpni0qX39kPMIukRmXc5UN6ojDkAcKDyzD6wiEl7UNO1SCrvV/KLfZSpCZe3I5M/dFNIzgx95LcdedxBbJ+p/mDbhJDu+O6HPhXGRroqBNUvabRN1GMsmZgg02cbd1ba3iOdFR0j7X4nxmh27nBHs6RN0vPEj7xC+0dT6PqM7y7kK2xA4I6EeMlqKVLfRjL9or5Byp6zFLNyO8T0zAg1DRyfBE4E4EAPowEtMBqwrbj6iGD0zXn85nQaCOp7bkMhp/HdwYKtxTKlNysp4gW/5QuW6PWC22WKTP7V9q6AFuY2YbaQaQ86Pyz+azuwh5jxmf1Rj0WLc6s4UStl08L6Js55QRci1dnnz4gw/ip4rg4+hsSzXbzA0iS2HbkR6N92uh+QmgBwv7O8u5Ex2WYRbD16X3djfjkvYYtyZTbPsYjt4qUujknMCO6o62CGzstW1aIZNGXaim21Yhhe3AHw4k7K81oB6imjx7uLjWxNldVmTkgiK3SULorjQAaYzhilQx3A3T++qd2g8IaL+X26SDlGBx6Uy1UcO1BGr10Ljgt1xmi9ZQfGnFkuEg2hqvLa0TjmCW6b2C4JPDzn7kXEdSJopydoO0IHkAh28hUXN1xbmd2szGLBM6Mxo7MaUxO4fognHUzbDgCfXWeKo9cpTgakOKYJr5sC1v7zuxwPW2hy6hAUVw43hXT5vPdGSyQfYz6X8Vs5u37OhXzROTP+lJ//Jyi0a1lvk/UxAhkcArqMQ55o2iM8tMDijCfXXDXakYeBTr5JLB8yfh9i+NrhCZT2J64pY/bF2YyLW2UIXeazToR331ohsSwXtFEPMQ374ahibqRN0TcAAr9pyjNlMaWjtibckISe70NJnyFJA5zBIU0F08RfdWGiSLMVvVV3Qcj8x8L4Gt8iYWRDoY8RuOOwD2Fk85K/VwrrvXHIFsQY5xnRRmjPta005tc8VagWpMEJsWeQqgvgLt3DS1RI9g8fgxv8CFjw3ngQmRqH62hoSzTUbaAR0C6g5C3JrhM9rr/nohQ+S1kdE4nMaz0LM5GYxAlDyBnWbaJ+axes4JAZVS4vw6l8z4XQ+70rYhfJQl0iJMNS27fxSJxnEc80odA7yF13K4MtIl1yvXHxOZdiMpOZbKbMTLgkiFBZ07UI9BbdIxSydgFhbXf1ujcavOQYS2M+V+rMZQPMDCKF1NALTBcUmrvH2HeATLorzR+Nc5pq67SHV8Y1ii5gWu0WPGH6uO933FsiodfpnbLQ3iEBrnVKJw1r/d0+Pu19GCmVxGCPbETWFeit2yQ+FbGiDp94YaguznIrxLfdAdbIBb4lB03+dRuz1snTdcLrNlhbnkzqTFcZbAxNB96SwV9IJ3+BjsD/yIRNpFwa9VdvxF52Wy+NZWqGt0ORIHAx2IEEng1Gj7ojNCHfBZW/bsqYrvxVo6N8xRKl9CelauG9yE5AogC2K7sjRMHabVdEXb/tRk+c7lMJpzzJjXvymT+ymsnTQDoOZLhsYvpe5WYg5s9ur3ozdGNIIZ8pimtl5aUUtm0hL+G70nU4oELbnaAKbXcCPTfN2VpaOH07fsrGum/iNDbfQPqKTOI5dqjQR/o23GDSnWJ9TSv60ltJgctYqhRv1qbC4dbEsy40QVMEMu1+gFK+qy1y/AYEnbtHqPemvnblbkgszw1x70EGrj6oJj28cwu881dJw1fWrYPHixRHp4jM/AjQBhzJBxin5R/jxui98au3w7zyOlC5p9gyUYhi28SJfIC2OhqgiPJoUBH4okhp9k5vuf8J63fZspjzJY1vVLLOgLgKWWeZG1fIOo+G2MCARsIRKxnq7+RrL9is7OPcCJGCduefR8fDlp5C2mp7RggwdwT5rFMhP/O7AW/3if2uqJT0TN2EEPk4RHc96SN9gBE6miDg1hEI0Y8oryMd05ZmdwSFSPMDWhwMU36PuqqOJ0kXYIcqKYZXAUYfTVG34AhE50dFsqSppuOrFDiLBb1hWY4vRkRMF1OQuNLeRyHwwsdNVHW+Ao358hq+L9/96/GbxXoGerslfzBrLWDWgDgNkcR3WoRbRYimUK4mVSTVP4yNIDdUaFqS2yTZRiiDy25VklqEPxYhMo+XR1URapP4YxEif34MqhPHJbyJ1Y7V612rTwqKT+g1TZXNU5bpEzxaeeATKwo8G6rmEp3u4+rp1s7qR4cbc3weAxjFcSY1Lj4afapnVCtcw6JYxjRTV8+6QYSL955LLM8EmdwxLhsdDyrmpHZoEpuScAy759+Z9h7tvu/w+f84KoEB+2y0WbzCMMUgIDYeeLJD4juwj3g8wBYCFPGRnrWd2GIncD/2GFTQjlOmu7HYRpRMpTUTM7Yf4Oa35xDHdS0obGyUR1DYNHm/GVNaS0//lglOXhY6zVYpxfr1rLKHGCibXvzK9pHjEQJlHf9VUe//aIW+xAJUulwtlL1Urksc3Cku9/k4cE7heIyNI6D4OJ5r46hw0rO760/0lit4FbscHvRsFa9SdepjNIxz3INBa0pZ/GJX6JW3PqWxsk0HRVWORYLAArDS4wkqzBzDEpd8uC5wV1cwrJlKTe+GtjpkI08/gFSfNBDQ5wRkZyd0TfWajJ9k/cvZvZIgSGn5+KSEkVY7PgGJIhtch5MzZI9OwFj5iUgXdfmzdum/nsV4vVZCNGngItt3FAMXuZDD+GSIIvCTIbx4N7UFkbeKnpmI6zuhbNOKpM9QA2Eg49PGFVrtCci/T+kXer3aHGw94PmMG1BKENdWA5mlOKGtDN9aIVzucNpBV+AUZPynLLnTM2/7AuQHJTnmDNezHJ+Elu1jthhiB1EAYoLTLr42oCl2epcu775sWZTyuu7OS0JBmjxbXTOcoQQu8U28N9e1SRh6oIBzeoS3Fh/BrcXrmVi87o7Ri2J0VolYG0OWrJSekUvCQB2J8zwPDDucHqNg7RT0jE65DF5pzrVu9kRqSpv8WaXBssthLKQX4tiRCe3aAOWUp6AvMs6JccrzPNsw7Q3YLf8t133vV7fKMpazNFXgar5NAgfDmGXyAioppzhzOQWV5lORMpkI1JeY32y0O2ZCmZCWbptE2Cs4QTnLA4V9gYBBp2BA+rT4THmuy1ZP1MK9uOpmqFyZnYc28gJ2RJzABh76oabRAjWNRza+/rikoVuHj7cOew3E4tNrQFELY9geaX+7n7895zK65jRRyoumTSycf5dI5RAy8/WaKMXrAexsj81oIhK9PeEFIA9NxtMiVytXnuWovagwDGyoU+TRe8Cj9/isfqWa5t94/I71RZKKW66MHlkmCRSn7xPHNJ0I6gNNkfY+VPRxpxfcvUj7PuU3IuZK1ZJYZoAqY47M7j3Yv+/h1Va9VjVT6RVzrvsyP6dW1pLnrxzDU5IbU4Z5Du7MuFBLyLldPSnC06HctlCujaPmHgwYylhqUf4u4oYtHiYo/2VMB9+D+fmGI1ZLeZuUlXF40PUFLk7LcJsMO8hGH0O3y+arnCVZzvT2nWo+QRfFF4Ug17MtV53Ds1wLJtSn2CkC+oFxyo0eTa7/A6HNL8nX3RKxWCvEYUEk4w1b4cyINkp4kHcfG4lBxUgor62txRZr0UdB+uVZ1eL+Qwvz6cIc43rPGAYJmchXQreO90tFacZSTAVpByX/HcpDD+zSbMPVL70J8ojnTiUeZnph9M4nfYL2bfUmMMjIVwXVG+dwwTJOeKFQOHolyiFSJqf8MILlyima5ekBYoVe8b9M+tIiXWp5Q/59KBYocp9YdoTAJm5EIC1X7xyHdOcVi3Grz3YV5suXVN3KZZk4dvZkMOf6oPXdx82oPgg/+nROhabtehJtl0M8T2HtqvKG9xuHSNaH1eJdn6ZlbejPw5Ql85VxI7/85++INXjumtaUL0X+REfZb6CmRB+0APsiFfO50I2mF2g0jeiM5uqQYhiYap8pqsB1+i0U5fdbUKUJrdPoW95r/E0klVSfuFYYIZJw1yGBY0NRYxqUfhuKOl6IW70G51nL6lc8k55VSaxCErrKTsGAmJWl0/02yqv6Flxv9916GH9+LY2XIil5KTRqZKvLaCM6034XxkQLuqTZvG6djW5lb9kulJTjAIn03bfYFLlebRXBvKfXBGpBTI/9c6iWeMFvWaarvGg7BIbnBDJQVQhmZaoQuADJ8RD+tED4oxqWJ0Y/2thsMTYdVFnoXwJRs//ldcGMHrPcY1PKRpb11PcRsewAVSSsiLhw0rJ/hCbt+0fQ/JRrauN4yw5fvUb2SQbrkwzpFBC/VU7hV9UTEMsyXVCi6yut66Fis3QL5cf26BRxRPVPIWtRae37dM4WItELMJ9lj8bX4uYT9s52OR/pW8qMl2naAF/U7yFG5n6vEglxHZrutnluTa+FMn5fboR11E0dXjUo7aHBiv6kooqc6rmKJwNA4pjlKh2jKwPWMFLmuSOzErH2cZkPbnq/o8m6dv+oRvo/4X4M6N3NHV/QzxidI5Xg4fXV0oJFEAvZx7N1l89JJfQUzD4ufYDLRJ1KvTRnCVumQnvzZ3jzoVgsRXrPiYOY5G0P70+WDt31IYtXf4ArToOKhpYi1jS/e+tmGlOazCi9X0GNqX4jRy2zRpYdQe0MkXaGKCRmaVI27zhN6N+QAb66sVXKbxQyHKus+vmeuj0ncmD/4QwxTvSv4MX4Ui5b/p9Cb1reMcm+KXIc7zo28QIL017b5YYYuHmqP8Lx7gjGu0Uqf07CNT/LnpGWKOj1fCXyvGbnsjpabIXEdirZyHSMtDPF2rlHQOldy3v5+DQ3eqLgGEPmlZthgpoGhuVX+tYXOG2/qOSK9LNO3He6Lj1eauEzLqI4DgkCE5symbpHYQTd+gd8WT5U1XGndys8g9WwKBfN9or1TaFGXRbxQ4zyIJbtwsV5A4UmYYoLkV9bEjr525L8DRTQma2I8aOW4g+liFi1BgBFNqBrPtMxz76W4oIni5VQ6M6sgJiRY2GnKsNUHw4jDVqoLTcAYLByf0OLxo92K95qRibjzPUfTKlV+D5xfVeVuO0HIMgcdFCpdQDKSQO+ZBoZs1vQz6XocE/aLtkAHLX5AEurgy4qrQ66SBHaKO09/N+YFZ+oipkMIxPhBCJieqEJ1YIb0YNjqJa5FMmyoHpn6k+pZCRfZVndzOwTO1Bad/JKuZDNdNBDne5BD3qOsqr3WCXv7WKJG+ssZ+mCrjGGj8ibgevcnk2cwIHOGnP+Dc4qwehnqtn7obTPslgBw5hEihTVGryIRFZF0EMcFcGSNbupW135+2z62t2QnNJ8tZaXHeOwQ484FiZasgPiR5BoaTBClQT/Eh7r5CBlmSjSOTOksV0ZNa7xO2GJzsK2ZWEjNPQ6GMEsTLDaJubrDfL3yaY+0JQlRpuX1EXzvCbCj2wzVMDWJvHcSox/iU/0oLrd8vuBNZKKDTNoZnyUAQ6XtzSTj99Q7mTvjYeYQj39+sxvO/NX2JJfVfPbK0YfWfL0VrPbCxbH8onqShUrcEmAcb4HrkXsEM7EnnVQYHgGEtyzTqttrNl6xlJtrLcd3DO83vUMILLO1i+4l+bVze8V2VwZlfRKRlXMOG87xLdgy+lsjOz42AVKKFMcI1uXuHUpr1xHH1sP9LBxWpXlECCjH5910aPEv75Ys1lJSmfKCsXAJogAwZGRTmWGcthAZmrYgFqVf6Ga7v9r/3sjjdoJmiAqQbpI1iGJosoIzRBXjofAsQ7v6YX0pNOz9dTja6UkFgTEdDHc6sCy73ssDxrCQ05DMOQ0XPGY39zwhOn5pr2n0KSpjZXGi20RZfNliXv3LODLh70LpJsLaKliWrz55PdUpEJBGzgucfEKmIA4rhWBov0QI3CHXSjcm4IaZbLVLeTp1+uT9nbUGwhVX6Qsm90pgNCAWHjf8UEkded40IecYQsFqvZD8RvuO96rp07TTJnXKK2MrVTo4VaG4QgT9QFCiWGqefp2TxpGCPM3BCXLYcHSXBijci75z+lYcz3VQMcT46RQwOMHvl+DhIVSR6na6VDZLdVma2IMmbiJ2R/Zr9409evN+jNxC89T0/BOKueOJkslsPFI4GGkXySDHSuC2kKY8uEEmvQ0L5Z1/aq3TK3Is5m6MDoqp+TxBnsnJIEJt7QMR4hwePgBRjobcMKdTsR+zgrcIplzVVFeQHwHD06WsxaOCYt3wzHyxkPAZnzBpM1o0uR6s5awS79QXXp6ZkT7UxkhHw0IlIqsDa/fvfHnmxkyXZp9ulCRBzl3oQfJch247izSDwg3PgRkpV8JSmW6dZewTENrK0XSG8ZyZSGq5UYE40gOrHJNJzD4fzXQMf4L1KL/+jq/qntmT1FDW6xwlc2ziGfi3fMl/XcE8eSjM4QxGQGI2kisf0Na9b3WBBbzFU1lDIJrDD4xlYXLrkvcsgT3IGQMmBr5QMgaLPUzDPgIz4KOwPzUqJA3R0r0kC1Y+sharDd3pvsimwtcNHNkrG5hzIhXprUetBoXCIo/ArX50WcZzOnNl7sBMvmSKjsSHbPE2itelERehbT4AYjaAkDUbzzrDzhUbT0etx7jBnKDYxCKjKk8O0YjpbVTbzoieWRFwx1drDBs3ie+Mjluu8QPYEwybiOMwhiwcI+LRR3sRBuXLcZFvm8uinUNBYbjOwoliRc5oH427mBlAGjsuKRSqgXXa/qLH+ulTa8Vip4ySXLVYR6XBLYL2oLjLuLoGYP+7Vi+G70RKdN56kNnBcikOl/ikBBDEYgdRi6AiYx7TSTtJvSwsViL5Hegwn3Jy3BUziynClSkHPVxI7VLbvuBDxWESsfjHrwOLE2p0WMi0cxJO+nkMGUsF5+Vgr7lENvCNiokbmhXLg3irx8D/vpObIxpfEsXItW9l59CryvfF0i0Oh8aEbtuujoo+V4f9IUpRcd9GALfM74letUMlvp3sVQ7wcR1Q19ZNyNDKxhbnSGajfEZ9CtrmRLKPOGQLcgjDXttvLYMS4slXfBsVajrN2T6jCcJiBlU0IfjEXL5I5hUs3Sm6cCfV1Bl8TKlC4UQvNyC4vlKPTWsEII/nNYWOK3fE8QnH1ZdEdlSERmjqbCxUlXSqfjORumkmOEQ17EIPu9lRgIUgTqTV92th12f6q2nGrUcxwDxOdb4gydKcYKrDxMYKgpjIn+bDVBnKGOjOb9hukK0l7n4Jktlo7BvBogqzJQxvm/BKuoUx/MjaDekYmjdbpg3n3zthfOX52tdvrqyVsPziOXjBrFHQrtSYz1H+N0xGLUcx+KWXpcJQDGL+VznXzCKLJu5WUxvqcqmpxKKuSGx3BDGkee4tn1clTvTS+UREveTPISfaFK3tcR0UVLl+sT04M6q8QUu04GByPFntmCJljY0/7mYX69EjDs3VkhM3/UUUiTHgeOn4wtEJj8GVBqd7HNperl2zHua/P6MzmjC1GBe5q+4Xm37xLMrLftLXPi5rLRvcqNPaZoz6QbbRa53oz+KPtyM6GazIl0+NXS9RJei4W7NpgCRzz2SQlP4/FDEHxAh5LgFS2d38xWLY/YraflfB/7qnM9zcc9ZWsFtesR1fYW1h/iOGUEh4+gejLWM78rdLBvEypa48Y2MS/z9RZ62zAWyeYHXJjg+cSwLb3bbkNyCGGnSQrXNCbg9kyK9zjaJdYvyucgebzq/dc/QTUsplfJSqCEsmQVYDl6xV66i9E07CEKoC1QDnQAgUmtFF7r4uRPz8x/tT1J2avLgEZk7eCp/rQlRYRO8pHsCPHct+KEk7vuHURrfBwy1Lvpvc92TDnLdkwDIuL7ppKW8s5S7Z0jKoKI/EUuhk7L99uOolVKLyLwLWRafWLYNDcsRNiyABGiyovwX803+3gxATZosr4Wy7cI0ZUpsqZAUx4SaOUF10ckJ0Az9xB8joXtdjfgXxaAWmQyyZ0qHISRBiHdjOBsORqifU1TGmPSrsap09r+Fhl54rH61pLM6RkwnwARCTkAiD+aJkx42bmD+GqcB+YrBDoz27Dt79h4axZkAHOqEr0V60GNZzjRIeN+ya5srY1Il64qnUHyH5eZ6yHw5wQS9E+dZm7kfboe+E1vvxBmW+xmMdpOlXjq4bz5dXBd/0Fh8pOoQiUdsC9cQbYtYDiw1YRjdxKsbHNRH/clHHZOdTPztuETtYXcW8QTnzgB6NEl5whd0samQTsSM1qXSb5TMWPq0A/HxYHxDuUovZxE3wAVRyyS+F0FzMcWGfACjfekiuSZleomNkhvZq3uTcHrt+DKngxuBJxghNhlV0zd+x3Tm9pzMLblWR9ocmzg+3qrnRCSwPAfqBk1PTc6hbsqfq8OmPafaioR+LHIljQgiYoZRXSbhO5Ybwn1BkyscOV3BwlTypZ42R1dxn1KREguhELo7XrkGDlMI+sQKyimr73qZ4hWhU1BZny41L8nui+LWNzSuMWFeYEcYgCxvie0D/zI9Rf5lCkg1ptepjES0f3kW/ztntypnjGc6aDOFZxLXg5nfw1u3wFt/00zNS+uU5PGUZIoZZqcAijNNi7+HYPa1cE4lOSvpKFWgvE9MP8AbO5xy0SqkYpjimsUUjKVNE56zhTF+dIIH/8bTCqJkkxbci/y32Q1Hs9X9bjijTVqkbk2KozIohzIMAiK/QqOwUxDsTL/MmO7CPb+uPZGKupZvj/twESldahVWbxHH9MD4yHkL08FCwGVOjHP5wvLZm7JI2ePppiyhi/rFNnr+Zw/tncqnZHVMGb4l9eep1Au2CYlQzzHH0jmYizhnCftSsJi+NzYz6XqN+rO9kHxrOqeZoqyIhA6eqyuTQguuyzzvInN4DnKMZspznq3kjUuXPNHwzEfZ4QQtK7Q1F8Ylvi3Nm7rwwnU2oM0HPRyj7OIcDG9V5W/8OSVjoucn1Isg09FY5DmTvyWNOavRRhgpFHEhcRzIBno+QDCQ8wHUBMuNAV3ratWexuqIJkKpVZnlCKmnIJhNE+bf51McFwBK3HOaFDQvNBJhv8LIpp9yXrPbzpe3A1feS1K/wDXB5OlFDyHLLUCgeyHSWLP1b0u5L8bI4lxAchK6FhpTsOe5btzwGjSBRQK8aKckgQsdC5zpy9NLVFQ6hVuoMnErdLXvGRYnlWk2V4YnbJNENiaQsYkHQ9bLK5RfNLw9hh436eOAL1mqyaS3WqcPHWSdPoB07gNbs+T9I3xueiTvF/UxxjShf+Cr47olL6KnECA7sLhy1cD0Sw0FJPXITPCraSy9aHw7TFndGKwdEivEyXi5NCyA05JXfZSMXwEE+hVdz3Qbdt9hmiKj1zVKsSNH2Uggr0wUwjuDeT6uQHR7xdczOvusKaH2LTYeyTdPa4gnTD+w1bxDxmhSM//93/8fIXqO1w==', 'internet_urban_2019.csv': 'eNqlm1tzG8mRhd/7VyD0sPZGlBFd96pH8M4hAXIAUhrpxdEke8i2QLQMApKpX79fNjSzXtvo1o6oiRmKQ2RXZWWePCcru75vV+3z61/v24da1bu/qHX92LQr9VpXa9WsNvV6VW/++ul+o7bru2ol3xWTg3dqst7eVeqy2jSr0eS5Xjf31ei/RofVurm7q6uVMqXOSgUzNimZGEwKKflicnKqJr8+PlWr5mXDb72ZNg8Py3p0XL1s1GjWrjdPo8mvYkyN/un3sHxdfey+fbOzrOM4lPJlkzJ+rJ2NxqdkcyptMTm9UpPVY7us1GJ795dF9VStMfLNcPd5a8baZH7dehXCOAQbk7E6ld7y+csDNVmy26ZSx9t1+6mWrdWrzbpajiYvzTcbIY29L23W2vDBMigfxjHYXEZtEsthu7MjFvLQrte9hnI5jjqFHHxWCaOYjNmGEIzDxvxY3a6aTf0wmqyru9Hxc7OuNvXLH/ddzvgr5xyzD9o4q5IflyYnE6xNSWd5Jse0fmShzWrwkGMeZxeyN6V10Xutsh4nH2JyzmubSoO5Keae6yF/hrF3NrscUggaXwTWVbpYWutj6UwqJgsM7daxGi2q5xZ77L6z0e3zvvm1uf8We8mOS++MjjgzmuiKyQ27YkuP22pUrR5GB9X6bvswvD89LrPxuCso48axTNZGb1wph3O7UJPti2yk6VlLtkRrzqUWZ8ex5WfBxZSSKwM2bnY2+r3D53iqwR8pEmxd+FkdQ+nwesn/KSYfjtXka02aNn9j8X22ZEvyVQavvYvKp7G2JWevc+QrFwdH5+pgu96uHpq9ORTGZTQ6Ezha6TSOKeiYMKdTLA6OL9VBvXxsts8DkW8ih5RKcshp2WSZgw5W65QNezo4nmFn1az2rsKUY1/KbpxXjlVoEq8sfS6NS8XByUS28ZEwHp1UL+1eK5l1kMRgSvbAACcdxqyBLHau1Nqzo9MjdVCtHpfVQ/3ypBbtVvLt953YcmyT1TxZy/fJGMAoWMdZs4vTOctYPlYDZxxil0qpzE5O1qpo2VAw1hjczNEXB2dYqp7WVbP6IQCIpfalIzs57aRYNpYX6g2mq+fqRY1unuo3Q4nB4YVsAQ7rJKKS8dEFy74J0uLg/EwdtC+kfJdrZ/X6a/3YfhY86XNAhyXgCBEQTQjK5bELrNSB9DF6TvRyLoFVrbcvvRFuiMesAaWsSxJPxTh2BEbklD0hhycvP4ih5ms9tE+SIwYsEQXKmTH4Tj0zmkphbHEwvcXM+lmA5NsR7Ox8Q6HOtVckQ7tsPjeDYONYZnSSiCREyEY8Eq11OQUBHsmsOSG9rr42y0HgsuMs+J6zBkETTqHQgYX8vUzO5ICtAyUoWD20L4PWCPAylpw2HiFdktMa+HLeJTFEmoIWdTM6kqN5qZbVcw8eehVB0mwsiR6TDRFH3mDiaSus4F9zi9+Vgpg5f2UzqOMCxdqRao5Hv8Mf7eblS7XaX+4BBjKSnYckZcaS7CG7xOKDKw4nJ+r3CPp1V1zm9aft3ZL17rMYQZ1sICBUBacJNiqVLV3JYRksztQh6/nPMUF95FwAqGQlb6INOECAnawvDs+O1eJLswHHlyROL3ha6ngoIQ7eeh2jSvAInEqAAhhABsbO1SE4sKqXo/MXsdebNMKKcCwIrksTUgwYuMRAsxzMEaEQAlmaVIPCaMOuIAEYswkrM7Gy6qmQwY3LVMI+DKesvRBHYo0E1kFrMKU4PH+rDttNPXr40/nntlnX+6ldEM6ZvfPBk7VeeT3GP7iX2kY+4OPpnONhK2276jMjRAzctcqTSS7FTInSmZgtDq+O1JvDFo6pRkf181jiZfxmry1thWBRtk0SBEkRUpP5w3/x8dXp77YGzEhNgmCyDxXsmDOHtkLhoG0eMxxVu2yf74ZRpiNWIZcwVQ/fsjt0zLgreZ0AWqxNsfbcrgGGnuVQ8tKu5BHHkF4vjCaBOcXhNedV3bWjt/X6Yf9hBXLBWVI7+gwBScK2ghFG4AlsLyuZE8Mt1Ws0l08NxaEemxKyFpyz3sCS2BmJ6RN8y/JjFnZ7oA6/Q7tILY4Sk8ELSTKdJfZoKHP8uCQkb99haV3dV+2gECLnbYR1a4F2W3Jc73Fw9fqMP37LzCEbUkoO31+rw9dPQ9UPagaoQBoCPGoncWKIUXh05LhYO1zx8Gt9/zRAOmEyAaR0TsNmDE6QuhqDM5Qj5IMtjo5v1Sn1r1q99hoSgokeIOjIKR3koHCD1KaQMiKqOPrpXB39rbkD+5s/Tm2o1eQUuKwlkiJcFIj0YAF7PppO1FH73Ky+I46onTYZIR1dSMKUIHQOECCyiqPZhTqqV8/V+mMvPidCLzhRRbAF1zkBPhJchmmXuLU4Ist+W9E/1ZxhwRXFXpZ0AQYcwis4SgjQVBx9mKBdH2thmn/ci57iDEB0xMmMHRmNVCTVdfTF8eGtOr7fwhnWg8wJniBIVQYKLknOWVAOjk/fqzfHj6+fWFQnaXeo98dXGwF8Q5xaWTDcwsmSNTggxTggA47BkON1s1nX+ymCBjmMlSIBmvkxlQtctaBZEgOLa7X4JKR7WMYHbw3QE+WcoLIZhIZvwZ2wcqOOXzbtgA4WUWSQP0AFZG3Hij1ZhwNDkEgqjm/O1PHmqWk/NT0b8uOcnHRJojJszgmF9EJUYAYn5zN10qyGKEZiC7AJqoIU5pi7/HdWxBqYarUrTkjbk+ZvTU9pt2NEeMpIha5DAoWUSgOMFidw2RPWfV/3LoJcZO9oAJybKMcxjQU0ovA30C1g50qdVOu2/j6Wg3DOHTsQTog7TxZTib77dbuq5ddGJ/XDeLTYvBCVe7dlpT5YdJRLyhgYRiZJUFPkdnE6OVCnFL/97ILM8L/zJURPiej0pBdI6IrTg/lvLZ+LZvX40PZraIMpAiPhDLIsifYBXLIzcnJUm+L0+AqMbtePg/0s4gUT5HrIAB9gGoTTZo4PjMHQ2USdPvURbTwh7LNDEONIe5FfcLji9ByXNHc8cVOte49Hytwp4Xm6bVY9+WrsmBgABbUWrqaNtB44zSAi6XR6oN6cVh0X2mnZvTRPi86nUnKMkBHg2og6yoR9cTo7+LaKvxw0aJrt/lwjGIw01QzoY4XgecRwJm2iwczP6vjv22rTAsrL0cC+ULzINeIKL6K5sAAfh80KvBen80N1uq7r/oSJHp0H/hEFyGhUKwmjE+QajUIGgeGn8yOx0ymUIR7kxuQ7stH7zs9drfHoSJyPnctuPYNA0nV2OhgQEQ44F6c3U5xbbepntOKgLHZjR64gg8GikvBysiEDyhpRtCzktrPWozlVRmJEaaMYzKAm+cx7PvNaDbc4iWZ8EHh61xhyXavZwadKn4qzC/j7Gfx9dCH/WkzmatSpnR7wyMJNrDQ1pEmRYhf0Z7MjhZ0HKOUgGUQGW9YAsRDYIAVAVoqX7sq01InibA4BX7eY6U36KHo6dxrcc8SuM1Rag1wC4Evs3Jyrs6qBlA0tiZQPTkQb52N2AsPn7DHli7PbmTrbrh6r9esA37QizW0ofQntkAwQpDZR1GAoc3F+NFPnq4dvSL3XwS6OeTy1n60haHwHSMLoSEpsu1icTzH0At9ofx1N+9ulsh0JXf7IZmIqzjkqViHV9196FQZghzuQaU5ZkADawweS8FCeSbqcr+vBqkuu8EWRJNLgTF0JlVoDlAIGRgzN1JtzoEN1Fe+5uf9REiU0zyNJYc8R5PACIobwTNJ4cR7Nez7/mcVXf//jzyCL4WNWCLm0VDtik0KiuHvSOjpTnC9w0P2ggzJO5iRR44QKzEhaIVlKr4PxkwEsdjHndNdVvfzjy0VFYTtFCFwi0a1kbeAAIFD8Q1AW5zcTdb6plq8DTV1yNGrEf3SCIELnCG/wLCT8YIufJlP1U/VcfY8uQZA5SAP1GdnsuzunQITxD14sQcKfrubqp3b98CO3azxFfFmyWdk08jcHZIvhAcVP1zMW+0kyZi+6QRLBEFJPUwcpiQJ3OgghIYU0Ri4mH9RF9bX6+NRdA/ZfVcgJSHsMOpGcMCfYNHmIBqUshuLieKYu6tVrLymRsoca4CR17LoVWioRmzIavlNcnLKe1/Xj69f/VWF9R0r5UdaNE3Fh4NQ+ke7FxZlI+ue79qEPmcASKRuR6uUoRILkArSGvAaZDPs5n6uLhlMnEnoATkv/NMM4osSAEeaGyJJ2bC4uZhO12Ixhj5vNS9d/n9Wfm8GiQmBSlBAdxBUucnKvqZPNqBFcRFy9uWhFQn3Dmv3nD03AKZSj5IzdNWWyhLv1wi1Z37sbdbH9QlX5oUsML22IINqeTXcl9HJyxSbb0fXRvLc2lN2NSumkaGXp6qQSoMNScXkwU5f1XbVqfyB/2HAnQCFvwu677r30o7QvLU+Yq8vmrlPoe/kkwChZnYOTgElkUYDtOGrI5cF7+fjrD8h7aWgIrUfWSa0hEuWKy5TZAsGXh7vYudzef0cjUUvbFgbiu7tRslKAKRDQCUvnx6y0vn/a1KuXTT0gnrtmNiCZtdJOVKdoZrnIk0O5YE3rZnRZrT7+e8m1kA0Wj8614jgnN0j4iqiwuGtBRNQv7eZp/+Wfk5TUUfCUdLIiEIP4B9ABc3wuLm9u2cnmaTs0EsDBe1KGmiZUSCp3HIsstCg01CrLuf1FXW7/UYMS2/Vjr0PiWCi8rMmaaMRB0E65folyP4aptxM5ns8DKxLKLr1wR2pb0XjIAmtsQB8CntPJIXEkLczvI66YA2iy9ECDsOAu66aTky5kptVaouXPJyiL+6fRJ/7639/V1Zzy6Gm7bu/v2x9IOicOs6JYQQVEAsGZ5E4GwYCgCr6YHl7xmFXFU3rbdrs1HU345eVD+7mfP1vhNBpA0crJ/UvXuum68pg4VVMk1mP1co/o3X/3DHMXVuFtd43M+nM00D25R50evcXG8qH5XL/8+8VYHMvvsukoD4cS8WQHmTek3/T4FzWt/9Hct9+B+0FkL5+WepZFL1rvM96zwCSmzi5ZxPrlqVr+00XSXnzN4yjqN0v7PYwJNCNq1HWtq+nF0bc7MaKufmiHE4oy1I0QUYlAw1COu2Ei8BU9aVnb5bk4aP+YAriEcgBLnFdGNmbk2kPUOXJienkjn978AJomlEVMcrcQZfpIi9DMOK6UNiBimT1PCW905rN0Pvb2k2T6RWghmyQKoErGSZaiJtjk7FgiFyCtH9ftQDyCDiGwwZgCYBr8GOxB3Um9g8hh61RsPba9Iyted2qZuExGJhGExaLCCAfIuwfJprPr3UHW65VkfoOOHo6NHYeWZhWHB1dna1cfWM5X6dX8fbv/okijTzrCI20I9qhZC9uRjhk25nKK23Wz6QB6byT4cSldzGyFMlFapJsa5VoPILxdfDPRbF/6rqu+uQBJhJAR9gYrziReQu96MOZdF47Vl6av0wwBDHJh3F3pWaqFC/AiJEExfb/oPv/aK28F65KwvVLiJGm5Ipd7u1jCcik8ppihKWYIw7umt9Uk926Uvu6GnMdnGQgxqJRczA4v1az+AlYsf0vTvYdKcEhGBdGogj7F7HiuZs1jve5tt0vHEpYjTkAncwqCGVDN2elk9+m+s2Ttzsm8TTe9lIPW1F0nCBaK2fkhBsDc6nFbfUcrBW5KVDgtliQJA651uIKNXAJXtQT5YF9ZCnQCB0rhDnL1yA9Skn4TPwWVcSkUmpT5Ur0OXRvJl0MT4xQj/Vxpcye5gpbDLmbXcjSfquW/UyFplYdQahMk8bNcGhgZdJAJptn8lpBoawGQfUcpYxlWmvXfpoJmH3ZR8KGudqJ8b0SKlJcWBWo8sXFqETQQ1QdTuZrO1NXzjwhSuRLx2XZnFLvbkFLuopykcHE9uVC//f4PTZTKeAtmtVxKSn+/a11bLdemIfOYGY9ZodMH66mVGuW0dAjJ0Q6DozfwbdtVieKa9Liu19vvuD/L0iqwhFSWCVM8C9Wglln8rEtfXFOZr5+aZfPpU7Oq+4qyjCAAMkJOS0PCexkt81K4iA92XVxfvmN/y2rbk+hRBn9Ki65HL1APimuKyXX1aVuNJEq+tbb3fl6LY5zMVET5Hr5BcSGNKXOYumIr7WBvrESjSpKlKFXNdj4KMLBu8kAu+K/n5+p6W683rUwMtKM/3y4GWShEKsIubLYiO3zHthEe0suULy9WL36Xv93Ex3XdflrWf3oZEsMyygIBx4p4X5qH11SrawJz+0gG97fxKTGguhRhGJRUH62F41FygpY1vcf5Hci9DsZk6i6cCEvZkBApJ9eDEpVdF4GgXByrdzW7OEBldU2D0+pr9UNNJDQrUjB2uZqEKAHzQcutvA3A9PX7E/VNK3DwrwMt3a74QoqARGDel6n4eXKjfq7kPulHugiBSKZilsBX2g1Hy4BCDl0fNBbzq1s1b5+HlB+8q+NcXlAJBdzJ9my1RJNcvZpiDsOYb1/4xEouGOs1B9auBoYYpUFFbAfkVkiCzrjUSM4SnCYU83cTNf/CWfUUSpl+EibXXVPm2M12ge+WJS0mt2pRbR+a7h7+RyYGcnffF0U1eODBCFIFm2X8wUiepmJxNGON0pbct1LpqLkI00+eY0aiFotjPiOcV0rdPhJjRT6F7t7eyfioiBaZPwN9i8XptVo0q8fqE5nbU7wgENLeZKm+JEm60re4PKC+LtvndjXMa42MHKQkDQtlogw1a4IH5RnZxeUxi6jX62p0Wberuu/GPmpJF9P1JGIJqDkHPYXoYOWtOl6OFtXy83eNXoCTmUgGFaNNMinCCrWDROhS7p1ysUCVLKodd1+139HB7l62yHE3Ot612o1mXVdT9WZBgix/vznvn2KT5rBmCaW2ojK6FqyMJVnAoljMcXq9vhtINnSvlnkmCLwQrK4PxQGSaDIKCbQsFkffyFF/zMlErN7RLekdUVekMagTeIuHboieqh3dtM91B4jXeOq++bT/BJ2Dp0RKtXHdAJmWi2WLrwTuFrc4HIUBhRicp6RoyGsP1Pwow1slUcH+rO5mxxZvL9Ri2X6uPn5XtzpJA1FYoe9uYJ2Ufnm7xCNCggOgMTjrDA69odHRDxAN+kXd1lFu1TqBGmUwDjpRLN7J+Cro1t/sc8JopSMFxzacn7zcUCZqU9lhO65/90Edv3wRJzV9qY/Qlns5KyrfygijXLojaojLX6aS+hviu1ojnEd/Ptpu/l9dqcX7Q2Lx9f6pXi7r/ZowBhnz4VtSn/MnkrIMTcjNu8MGZ/4q4vj3Uafdef1hpJVKAzwiBeBORH8obg4n6ma7/rjr9B9WkJ6X7x4qzNQEBCNSzMqEm8fakcwMP/RMPsstirz24bueL9lcgvwyf2WLm9MrddM+tn2vawDxtmtzOD120hwFplDVPPqMjTxVTb/SkNe1vLBIQhdWnSQIs5CrIJc88rrFzU8X6qb6W/NNEvQEotwMUfe7wVx548NKE9k7GclmLxfTzq/y4tKQoe5SwclUkUHK487i5nKhbprndv2XS0hVT/ExovdkTgvaaYR9cLZy9etJ1uLmaoY/KWC9nRovo56U9G4QCbmpPT8J0Jfi5obzAHKah+qhC4+b9q56HOwIRiGHkA4Opht+tnKPg1s4uVzc3LKmLT75EcIgp9jVke7qy7lu+kj63fIaSSB7eMy8c3/zWg8QrxxZKtELAOlu/DxIhMows8x7Y+gthj5Xy20/uTTUSviwgW7BPW4+EIzV6mtvR0mHsZdBbvhjp71l1NPLDDX/LW5PJ+r2sZecaeknCTuT+xOqrJW5Ky29iNIWtxdzdftRXvXpd0A5ZpcwBWlMmWClnQvpjEQhmEo4F7fohNv19ntkgszaeXm1JcJ/vUtC7NgViJZLjodjuV1MfhtOW2y6FxH/wysOcunFwSYSIcohiGwjucSQ3AFmV9x+OFC3X+/q4RyVF0+6SGFDqOkk1MZnsxsHkl558fbwprt/eEt15uNdnG+e6tFuwqnTxcNCm6izu1d8EP4oSoAgZ3xhirew0DdvoaFft/VSbj8PBl+NgohKKxeWZMQXaPayezUsZuLx7emBOpBG48sTS14/Nt8/Ai6XxzL4GBFBVOFQvD2fq/9rA707XoyHKxwHW8rQhWh5TgYvzqaYqjejWd8YVeiyy0MASpAbQmhFk9is5eJYBt6Kt7c36m212labvnwzdgwdkXG0LOzPF+8W1OuBVzp1Eg4LtXdyvQpx+eXiF3XRvrSf235wlrIs5Adq6AOV5v0xpPV9DbKrHx2c0XDRLJTYGPFFFtqPxrVCZYoPcjO269HtbO3loejQjC+kwRWkvS0iX7q7pF3xYXqgPuyGGfcCiUgnnSRPhBZ1c0/ybpZUkuIDpOxD83xX3X3ZT10pfzLVJtPCMuYnN5hgMUU9uOJ/AE3+5ow='}
data_paths = {}
for name, expected in expected_files.items():
    destination = Path('data') / name
    if destination.exists():
        raw = destination.read_bytes()
    else:
        raw = zlib.decompress(base64.b64decode(saved_snapshots[name]))
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError(f'Changed project input: {name}. Review its source before continuing.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw)
    data_paths[name] = destination
print('Verified project data:', ', '.join(data_paths))

Look for **Verified project data**. The [data dictionary](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md) records raw API URLs, licenses, transformations and limitations. A frozen snapshot makes the exercise reproducible; it does not make the observations current.

### Describe the request explicitly

Build the World Bank URL from the indicator code and query parameters. urlencode handles query punctuation. This URL describes the original API request; we read the frozen response below so a normal rerun does not silently switch to a revised dataset.

In [ ]:
from urllib.parse import urlencode
indicator = "IT.NET.USER.ZS"
base_url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
query = urlencode({"date": "2019", "format": "json", "per_page": 1000})
request_url = base_url + "?" + query
print(request_url)

date selects the observation year, format requests JSON and per_page requests up to 1000 records. A server can still paginate; always inspect its returned metadata. No API key appears in this request. Other services may require credentials; never put a private key in a shared notebook or URL.

### Inspect the actual response and pagination

Load the pinned JSON bytes. This API uses a two-element list: a metadata dictionary followed by observations. We explicitly refuse an incomplete multi-page response rather than treating its first page as the world.

In [ ]:
payload = json.loads(data_paths["raw/wb-internet-2019.json"].read_text())
response_info, observations = payload
print(response_info)
if response_info["pages"] != 1 or response_info["total"] != len(observations):
    raise ValueError("Incomplete response: retrieve and validate every page before analysis.")
print("API records:", len(observations))

The snapshot has 266 records and one page. Its lastupdated field is 2026-11-11, while each observation is for 2019. Retrieval date, database update date and observation year answer different questions. The API can revise older estimates, so the URL alone is not a complete version identifier.

### Flatten only the fields you need

A loop creates one simple record per observation. countryiso3code is the source economy code, while country.value is its display name. We preserve missing indicator values.

In [ ]:
import pandas as pd
records = []
for row in observations:
    records.append({"economy_code": row["countryiso3code"], "api_name": row["country"]["value"], "year": int(row["date"]), "internet_pct": row["value"]})
api_table = pd.DataFrame(records)
print(api_table[["economy_code", "api_name", "year", "internet_pct"]].head())
print("Blank codes:", api_table["economy_code"].eq("").sum())

Five income aggregates have blank three-character codes in this response. Other aggregates have nonblank codes. Dropping blank codes alone would still leave regional totals mixed with individual economies, and could double-count observations.

### Use geography metadata to exclude aggregates

The pinned country metadata classifies aggregates with region.id equal to NA. Build a lookup of individual economies and join it to the API records. The metadata were retrieved in 2026; these labels are not a reconstructed 2019 classification.

In [ ]:
metadata = json.loads(data_paths["raw/wb-economies.json"].read_text())[1]
economy_rows = []
for row in metadata:
    if row["region"]["id"] != "NA":
        economy_rows.append({"economy_code": row["id"], "economy": row["name"], "region": row["region"]["value"].strip()})
economy_lookup = pd.DataFrame(economy_rows)
individual_records = api_table.loc[api_table["economy_code"].ne("")]
verified_economies = individual_records.merge(economy_lookup, on="economy_code", how="inner", validate="one_to_one")
print("Individual economies:", len(verified_economies))
print("Missing internet-use values:", verified_economies["internet_pct"].isna().sum())

Now there are 217 economy observations and 28 missing internet-use estimates. An economy may be a country or territory; these labels are not a claim about sovereignty. Excluding aggregates and deciding how to handle missing indicators are separate operations.

### Record rights and version before using results

The World Bank metadata for [internet use](https://databank.worldbank.org/metadataglossary/world-development-indicators/series/IT.NET.USER.ZS) explicitly identifies **CC BY 4.0** and asks users to cite the **International Telecommunication Union**. The [World Bank data terms](https://datacatalog.worldbank.org/public-licenses) include additional conditions. Publicly accessible does not mean every input has the same license. Document provider, exact request, retrieval date, observation year, source license, transformations and exclusions. The course's original material still has no selected license.

The committed raw response is retained unchanged, and a SHA-256 checksum identifies its bytes. A checksum is a fingerprint, not evidence that a provider's definition or measurements are correct. If a live download changes, keep the earlier input and investigate the revision. Do not simply replace the expected checksum to make an analysis pass.

The prepared setup cell recreates this notebook’s bundled public-data snapshot when local inputs are absent and verifies its bytes. Bundling the small, licensed snapshot keeps the browser exercise usable while the course repository is private and prevents a live API revision from changing an old exercise. This separates reproducible learning from a live API update. For real research, plan an explicit refresh step, review new metadata and preserve the old version with the results that used it. Check rate limits and reuse terms before automating repeated requests.

## Try it

Work in your own copy. Add a cell for each task and attempt it before reading a solution.

### 1. Reconcile the missingness with the prepared project CSV.

<details><summary>Solution 1</summary>

Run this solution after your own attempt.

In [ ]:
prepared = pd.read_csv(data_paths["internet_urban_2019.csv"])
print(len(prepared), prepared["internet_pct"].isna().sum())
comparison = prepared.merge(verified_economies[["economy_code", "internet_pct"]], on="economy_code", validate="one_to_one", suffixes=("_csv", "_api"))
pd.testing.assert_series_equal(comparison["internet_pct_csv"], comparison["internet_pct_api"], check_names=False)
print("Values and missingness agree.")

The CSV retains all 217 economies, including 28 missing internet-use values. A missing value should survive a faithful transformation rather than being turned into a zero or silently removed.

</details>

### 2. Identify a changed file using its fingerprint.

<details><summary>Solution 2</summary>

Run this solution after your own attempt.

In [ ]:
import hashlib
original_bytes = data_paths["raw/wb-internet-2019.json"].read_bytes()
original_hash = hashlib.sha256(original_bytes).hexdigest()
changed_hash = hashlib.sha256(original_bytes + b" ").hexdigest()
print("Same bytes:", original_hash == changed_hash)

Even adding whitespace changes the byte fingerprint, although parsed values might agree. A checksum failure requires inspecting both the version and the substantive content; it is not automatically evidence of changed statistics.

</details>

### 3. State the sample available for the next project.

<details><summary>Solution 3</summary>

Run this solution after your own attempt.

In [ ]:
complete_pairs = prepared.dropna(subset=["internet_pct", "urban_pct"])
print("Complete pairs:", len(complete_pairs))
print("Observation years:", complete_pairs["year"].unique().tolist())

There are 189 complete pairs for 2019. These observations are not a random sample of all people, and economies with missing estimates may differ systematically from those retained.

</details>

## When it goes wrong

These demonstrations catch a specific intended error so the notebook continues. Unexpected errors elsewhere should still stop execution.

### The response structure is guessed

Our fictional response did not use the key data.

In [ ]:
try:
    fictional["data"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the JSON type, actual keys and official API documentation. A successful HTTP response can even be an HTML notice rather than the JSON data you expected.

### A changed input is accepted without review

This explicit guard rejects different bytes, rather than changing the original file.

In [ ]:
try:
    if hashlib.sha256(original_bytes + b" ").hexdigest() != original_hash:
        raise ValueError("Input fingerprint changed; preserve the previous version and inspect the update.")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Do not catch every error and continue with whatever old or partial data happen to be present. Make failed downloads, incomplete pages and changed versions visible.

## Recap

- An API is a transport and format contract, not a data-quality or reuse guarantee.
- Check response structure, pagination, keys, units, missingness and geography.
- Keep observation year, retrieval date and update date separate.
- Preserve exact inputs and review revisions before changing a pinned analysis.

**Next:** [5.4 Checking AI coding assistance](04-checking-ai.ipynb).